# Dallas CIP Parser

Parses all PDFs in `Dallas/PDF/` and writes one CSV per PDF into `Dallas/CSV/`.

## Key rules
- `previous_appropriations` = inception-to-date budget column ("Budget ITD", "Budget as of [date]", "Budget Balance").
- `project_total` = the final "Total Project Cost" / "Total Estimated Cost" column.
- `project_id` = code appended to the project name (e.g. `- VG02`, `- W221`); blank for early years that lack codes.
- `project_type` = the Service column.
- `end_year` = year extracted from the completion/service date; blank for "Ongoing" or N/A.
- `start_year`, `project_description`, `project_justification`, `address_location` = blank (not present in Dallas PDFs).

## Format families detected
| Format | PDFs | Rotation | Notes |
|--------|------|----------|-------|
| **A** | 2007–2016 | 0°/90° | 14 cols; Key Focus Area + In Service Date; no project ID codes |
| **B** | 2017 | 90° | 13 cols; similar to A |
| **C** | 2018 | 270° (transpose+reverse) | ACTIVITY + Fund + Unit Number; 5 FY columns |
| **D** | 2019–2021 | 0°/90°/270° | 12 cols; project ID codes begin here |
| **E** | 2022–2025 | 0° | 12 cols; Future Costs column added |

**Rotation handling:**
- 0° / 90°: pdfplumber table extraction works directly.
- 270°: the extracted table is transposed with each cell's characters reversed.
  Fix applied: transpose rows↔cols, reverse column order, reverse each cell string.

## Cell 1 – Imports & paths

In [7]:
import re
import warnings
from pathlib import Path

import pdfplumber
import pandas as pd

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
SCRIPTS_DIR = Path('.').resolve()          # …/CIPBD/Scripts/
CITY_DIR    = SCRIPTS_DIR.parent / 'Dallas'
PDF_DIR     = CITY_DIR / 'PDF'
CSV_DIR     = CITY_DIR / 'CSV'
CSV_DIR.mkdir(exist_ok=True)

REQUIRED_COLS = [
    'cip_year', 'project_type', 'source_page', 'department',
    'project_name', 'project_id', 'address_location',
    'start_year', 'end_year',
    'project_description', 'project_justification',
    'previous_appropriations', 'project_total',
    # Dallas-specific
    'funding_source', 'council_district', 'completion_date',
    'spent_or_committed', 'remaining',
    'fy_budget', 'fy_planned_1', 'fy_planned_2', 'fy_planned_3', 'fy_planned_4',
    'future_costs',
]

print('Paths OK.')
print(f'  PDF_DIR : {PDF_DIR}')
print(f'  CSV_DIR : {CSV_DIR}')
print('PDFs found:', sorted(p.name for p in PDF_DIR.glob('*.pdf')))

Paths OK.
  PDF_DIR : C:\Users\vince\Documents\GitHub\CIPBD\Dallas\PDF
  CSV_DIR : C:\Users\vince\Documents\GitHub\CIPBD\Dallas\CSV
PDFs found: ['2007.pdf', '2008.pdf', '2010.pdf', '2011.pdf', '2012.pdf', '2013.pdf', '2014.pdf', '2015.pdf', '2016.pdf', '2017.pdf', '2018.pdf', '2019.pdf', '2020.pdf', '2021.pdf', '2022.pdf', '2023.pdf', '2024.pdf', '2025.pdf']


## Cell 2 – Format detection (rotation per PDF)

In [8]:
def classify_pdf(pdf_path):
    """
    Returns {page_num (1-indexed): rotation_degrees} for all pages.
    All Dallas PDFs are text-based (no OCR required).
    """
    page_rot = {}
    with pdfplumber.open(pdf_path) as pdf:
        for i, pg in enumerate(pdf.pages):
            page_rot[i + 1] = pg.rotation
    return page_rot


pdf_files    = sorted(PDF_DIR.glob('*.pdf'))
rotation_map = {}
for p in pdf_files:
    rots = classify_pdf(p)
    rotation_map[p.stem] = rots
    non_zero = set(v for v in rots.values() if v != 0)
    needs_fix = any(v == 270 for v in rots.values())
    flag = ' ← needs transpose fix' if needs_fix else ''
    print(f'{p.name}: {len(rots)} pages, rotations={non_zero or {0}}{flag}')

2007.pdf: 314 pages, rotations={90}
2008.pdf: 295 pages, rotations={90}
2010.pdf: 250 pages, rotations={90}
2011.pdf: 216 pages, rotations={90}
2012.pdf: 232 pages, rotations={90}
2013.pdf: 224 pages, rotations={90}
2014.pdf: 196 pages, rotations={0}
2015.pdf: 180 pages, rotations={90}
2016.pdf: 164 pages, rotations={90}
2017.pdf: 140 pages, rotations={90}
2018.pdf: 234 pages, rotations={270} ← needs transpose fix
2019.pdf: 170 pages, rotations={90}
2020.pdf: 90 pages, rotations={0}
2021.pdf: 215 pages, rotations={270} ← needs transpose fix
2022.pdf: 212 pages, rotations={0}
2023.pdf: 174 pages, rotations={0}
2024.pdf: 198 pages, rotations={0}
2025.pdf: 210 pages, rotations={0}


## Cell 3 – Table extraction & cell utilities

In [9]:
def fix_270_table(raw_table):
    """
    Pages rotated 270° are extracted by pdfplumber as a transposed table
    with each cell's characters reversed.
    Fix: (1) transpose rows↔cols, (2) reverse column order, (3) reverse cell text.
    """
    if not raw_table:
        return []
    max_cols   = max(len(r) for r in raw_table)
    padded     = [list(r) + [''] * (max_cols - len(r)) for r in raw_table]
    transposed = list(map(list, zip(*padded)))
    return [
        [str(c)[::-1] if c else '' for c in reversed(row)]
        for row in transposed
    ]


def extract_tables_from_page(pg):
    """Extract all tables from a page, applying the 270° fix where needed."""
    raw_tables = pg.extract_tables() or []
    if pg.rotation == 270:
        return [fix_270_table(t) for t in raw_tables]
    return raw_tables


def clean_cell(c):
    """Normalise a table cell: collapse all whitespace to single spaces."""
    if c is None:
        return ''
    return re.sub(r'\s+', ' ', str(c)).strip()


def parse_number(s):
    """Convert a cell value to float; handles commas, dashes, and parentheses."""
    if not s:
        return None
    s = re.sub(r'[,$]', '', str(s).strip())
    s = re.sub(r'^\((.+)\)$', r'-\1', s)   # (1,234) → -1234
    if s in ('-', '', 'N/A', 'n/a', 'Various', 'Ongoing'):
        return 0.0
    try:
        return float(s)
    except ValueError:
        return None


print('Table utilities loaded.')

Table utilities loaded.


## Cell 4 – Column-header normalisation

Maps the raw header text (which varies by year) to canonical field names.

**Important ordering note:** `'in service'` must appear *before* `'service'` in
`HEADER_MAP` so that "In Service Date" is correctly tagged `completion_date`
rather than `project_type`.

In [10]:
# Each entry: (substring_to_match_in_lowercase_header, canonical_tag).
# Checked in order; first match wins for each column.
# 'in service' MUST precede 'service' to avoid mis-tagging "In Service Date".
HEADER_MAP = [
    # ── Project name ─────────────────────────────────────────────────────────
    ('project name', 'project_name'),
    ('projects',     'project_name'),
    ('project',      'project_name'),
    ('activity',     'project_name'),

    # ── Completion / service date  (before 'service' → project_type!) ────────
    ('in service',      'completion_date'),
    ('comp. date',      'completion_date'),
    ('completion date', 'completion_date'),
    ('amended date',    'completion_date'),
    ('est. comp',       'completion_date'),

    # ── Service / project type ───────────────────────────────────────────────
    ('service',      'project_type'),

    # ── Key Focus Area (older PDFs only) ─────────────────────────────────────
    ('key focus',    'key_focus'),

    # ── Funding ──────────────────────────────────────────────────────────────
    ('funding source', 'funding_source'),
    ('fund',           'funding_source'),

    # ── Council district ─────────────────────────────────────────────────────
    ('council',      'council_district'),

    # ── Unit / project number (2018 style) ───────────────────────────────────
    ('unit number',  'unit_number'),

    # ── Budget (inception-to-date / previous appropriations) ─────────────────
    ('budget itd',     'previous_appropriations'),
    ('budget as of',   'previous_appropriations'),
    ('budget balance', 'previous_appropriations'),
    ('budget',         'previous_appropriations'),   # generic fallback

    # ── Spent / committed ────────────────────────────────────────────────────
    ('spent',               'spent_or_committed'),
    ('capital expenditure', 'spent_or_committed'),
    ('expenditure',         'spent_or_committed'),

    # ── Remaining ────────────────────────────────────────────────────────────
    ('remaining',    'remaining'),

    # ── Total project cost ───────────────────────────────────────────────────
    ('total project cost',   'project_total'),
    ('total estimated cost', 'project_total'),
    ('total projected',      'project_total'),
    ('total',                'project_total'),

    # ── Future costs ─────────────────────────────────────────────────────────
    ('future cost',  'future_costs'),
]

# FY column: "FY 2019-20", "FY2007-08", "FY 2021-22 Budget", etc.
FY_RE = re.compile(r'FY\s*(\d{4})[\s\-–](\d{2,4})', re.I)


def normalise_header(raw_headers):
    """
    Map raw header strings → canonical tags.
    FY columns become 'fy_YYYY_YY' keys; the caller maps them to
    fy_budget / fy_planned_1 / … via assign_fy_slots().

    Returns (tags_list, fy_labels_sorted).
    """
    tags      = []
    fy_labels = []   # ordered by first appearance
    used_tags = set()

    for raw in raw_headers:
        h = clean_cell(raw).lower()

        # ── FY column? ──────────────────────────────────────────────────────
        m = FY_RE.search(h)
        if m:
            fy_key = f'fy_{m.group(1)}_{m.group(2).zfill(2)}'
            if fy_key not in fy_labels:
                fy_labels.append(fy_key)
            tags.append(fy_key)
            continue

        # ── Named column ────────────────────────────────────────────────────
        matched = None
        for pattern, tag in HEADER_MAP:
            if pattern in h:
                # Each tag may appear only once (first occurrence wins).
                if tag in used_tags:
                    continue
                matched = tag
                used_tags.add(tag)
                break
        tags.append(matched or f'_skip_{len(tags)}')

    return tags, sorted(set(fy_labels))


def assign_fy_slots(tags, fy_labels):
    """
    Replace raw FY keys ('fy_2019_20') with positional slot names
    ('fy_budget', 'fy_planned_1', …, 'fy_planned_4').
    """
    slot_names  = ['fy_budget', 'fy_planned_1', 'fy_planned_2',
                   'fy_planned_3', 'fy_planned_4']
    fy_slot_map = {fy: slot_names[i]
                   for i, fy in enumerate(fy_labels) if i < len(slot_names)}
    return [fy_slot_map.get(t, t) for t in tags]


def is_project_list_table(tags):
    """True if the table has a project-name column and at least one numeric column."""
    return 'project_name' in tags and any(
        t in tags for t in (
            'previous_appropriations', 'project_total',
            'fy_budget', 'fy_planned_1', 'spent_or_committed',
        )
    )


print('Header normalisation loaded.')

Header normalisation loaded.


## Cell 5 – Project-name and date parsing

In [11]:
# Project ID: trailing " - CODE" where CODE = 1-2 letters + digit + up to 5 alphanum.
# Examples: "Fire Station #46 - VG02", "City Hall - VH05", "AMR - P291", "2010 CO - IC70"
PROJECT_ID_RE = re.compile(r'[-\s]+([A-Z]{1,2}[0-9][A-Z0-9]{0,5})\s*$')


def split_project_name(raw_name):
    """
    Split 'Project Name - CODE' into (clean_name, project_id).
    Returns ('name', '') if no trailing code is found.
    """
    name = clean_cell(raw_name)
    m    = PROJECT_ID_RE.search(name)
    if m:
        pid   = m.group(1)
        clean = name[:m.start()].strip().rstrip('-').strip()
        return clean, pid
    return name, ''


def extract_end_year(date_str):
    """
    Extract a 4-digit calendar year from a completion / service date string.
    Handles: 'MM/YYYY', 'MM/YY', '4th/07', 'Ongoing', 'N/A', 'Various'.
    Returns int year, or '' if not parseable.
    """
    if not date_str:
        return ''
    d = date_str.strip()
    # Explicit 4-digit year
    m = re.search(r'\b(20\d{2}|19\d{2})\b', d)
    if m:
        return int(m.group(1))
    # 2-digit year after slash: "4th/07" → 2007, "09/24" → 2024
    m = re.search(r'/(\d{2})$', d)
    if m:
        yy = int(m.group(1))
        return 2000 + yy if yy < 50 else 1900 + yy
    return ''


# ── Unit tests ───────────────────────────────────────────────────────────────
assert split_project_name('Fire Station #46 - VG02')  == ('Fire Station #46', 'VG02')
assert split_project_name('City Hall - VH05')          == ('City Hall', 'VH05')
assert split_project_name('2010 CO - IC70')            == ('2010 CO', 'IC70')
assert split_project_name('Central Library')           == ('Central Library', '')
assert extract_end_year('09/2026') == 2026
assert extract_end_year('4th/07')  == 2007
assert extract_end_year('11/30/19') == 2019
assert extract_end_year('Ongoing') == ''
print('Name/date parsing OK.')

Name/date parsing OK.


## Cell 6 – Department detection and row parsing

In [12]:
DEPT_SKIP = {
    'OVERVIEW', 'MISSION', 'PROJECT LIST', 'SOURCE OF FUNDS',
    'USE OF FUNDS', 'SUMMARY', 'CAPITAL IMPROVEMENT PROGRAM',
    'SERVICE DESCRIPTIONS', 'OPERATING AND MAINTENANCE COSTS',
}
DEPT_RE = re.compile(r'^[A-Z][A-Z ,&/\-]+$')


def get_department_from_page(pg):
    """
    Return the department name from the page header (first ALL-CAPS line),
    stripping the boilerplate 'CAPITAL IMPROVEMENTS' suffix.
    """
    txt = pg.extract_text() or ''
    # 270° pages have reversed text lines
    if pg.rotation == 270:
        lines = [ln[::-1].strip() for ln in txt.split('\n') if ln.strip()]
    else:
        lines = [ln.strip() for ln in txt.split('\n') if ln.strip()]

    for line in lines[:6]:
        # Strip trailing boilerplate
        line = re.sub(r'\bCAPITAL\b.*', '', line).strip()
        line = re.sub(r'\bIMPROVEMENTS?\b.*', '', line).strip().rstrip(',')
        if DEPT_RE.match(line) and len(line) > 4 and line not in DEPT_SKIP:
            return line
    return ''


# Row-filter patterns: skip these project-name values
_SKIP_STARTS = ('project', 'activity', 'total', 'grand total',
                'source of funds', 'use of funds')
_SKIP_EXACT  = re.compile(r'^(Project|Projects|ACTIVITY|Service|Services)$', re.I)


def parse_data_row(row_cells, col_tags, cip_year, department, pg_num):
    """
    Convert a row of cell strings into a project record dict.
    Returns None for header repetitions, subtotals, and empty rows.

    Note: col_tags may contain duplicates (e.g. two 'project_type' columns in
    older PDFs that have both 'Service' and 'Key Focus Area').  We use
    first-wins so the earlier column always takes precedence.
    """
    # Build cell dict — first occurrence of each tag wins
    cell   = {}
    for i, tag in enumerate(col_tags):
        if tag not in cell:          # first-wins for duplicate tags
            cell[tag] = clean_cell(row_cells[i]) if i < len(row_cells) else ''

    raw_name = cell.get('project_name', '')
    if not raw_name:
        return None

    # Skip header repetitions and subtotal rows
    if any(raw_name.lower().startswith(f) for f in _SKIP_STARTS):
        return None
    if _SKIP_EXACT.match(raw_name):
        return None

    # Must have at least one non-zero numeric value to be a real data row
    _num_tags = ('previous_appropriations', 'project_total',
                 'fy_budget', 'fy_planned_1', 'fy_planned_2', 'spent_or_committed')
    has_value = any(parse_number(cell.get(t, '')) not in (None, 0.0) for t in _num_tags)
    if not has_value and parse_number(cell.get('project_total', '')) is None:
        return None

    # Split project name and extract ID code
    name, pid = split_project_name(raw_name)
    # 2018-style: unit number column carries the project code
    if not pid and cell.get('unit_number', ''):
        m = re.search(r'U_([A-Z0-9]+)', cell['unit_number'])
        if m:
            pid = m.group(1)

    completion = cell.get('completion_date', '')
    end_yr     = extract_end_year(completion)

    def num(tag):
        v = parse_number(cell.get(tag, ''))
        return v if v is not None else ''

    return {
        # ── CIPBD standard columns ───────────────────────────────────────────
        'cip_year':               cip_year,
        'project_type':           cell.get('project_type', ''),
        'source_page':            pg_num,
        'department':             department,
        'project_name':           name,
        'project_id':             pid,
        'address_location':       '',
        'start_year':             '',
        'end_year':               end_yr,
        'project_description':    '',
        'project_justification':  '',
        'previous_appropriations': num('previous_appropriations'),
        'project_total':          num('project_total'),
        # ── Dallas-specific columns ──────────────────────────────────────────
        'funding_source':         cell.get('funding_source', ''),
        'council_district':       cell.get('council_district', ''),
        'completion_date':        completion,
        'spent_or_committed':     num('spent_or_committed'),
        'remaining':              num('remaining'),
        'fy_budget':              num('fy_budget'),
        'fy_planned_1':           num('fy_planned_1'),
        'fy_planned_2':           num('fy_planned_2'),
        'fy_planned_3':           num('fy_planned_3'),
        'fy_planned_4':           num('fy_planned_4'),
        'future_costs':           num('future_costs'),
    }


print('Department + row parser loaded.')

Department + row parser loaded.


## Cell 7 – PDF parser (one PDF → list of project dicts)

In [13]:
def find_header_row(table):
    """
    Scan the first 5 rows for the column-header row, identified by having
    'Project', 'Projects', 'Project Name', or 'ACTIVITY' as its first
    non-empty cell.

    Note: older PDFs prepend a section-title row (e.g. 'CITY FACILITIES
    CAPITAL IMPROVEMENTS') before the real header; this loop skips it.

    Returns (header_row_index, col_tags) or (None, []).
    """
    for i, row in enumerate(table[:5]):
        cells = [clean_cell(c) for c in row]
        first = next((c for c in cells if c), '')
        if re.match(r'^(Project(s| Name)?|ACTIVITY)$', first, re.I):
            tags, fy_labels = normalise_header(cells)
            tags = assign_fy_slots(tags, fy_labels)
            if is_project_list_table(tags):
                return i, tags
    return None, []


def parse_pdf(pdf_path):
    """
    Parse every project-list page in a PDF.
    Returns (cip_year, list_of_project_dicts).
    cip_year is taken from the filename (e.g. '2019.pdf' → 2019).
    """
    stem     = pdf_path.stem
    cip_year = int(stem) if stem.isdigit() else None
    projects = []
    current_dept = ''

    with pdfplumber.open(pdf_path) as pdf:
        for pg_idx, pg in enumerate(pdf.pages):
            pg_num = pg_idx + 1

            # Track department from page header
            dept = get_department_from_page(pg)
            if dept:
                current_dept = dept

            for table in extract_tables_from_page(pg):
                if not table or len(table) < 2:
                    continue

                hdr_idx, tags = find_header_row(table)
                if hdr_idx is None:
                    continue

                for row in table[hdr_idx + 1:]:
                    rec = parse_data_row(
                        row, tags, cip_year, current_dept, pg_num,
                    )
                    if rec is not None:
                        projects.append(rec)

    return cip_year, projects


print('PDF parser loaded.')

PDF parser loaded.


## Cell 8 – Main loop: parse all PDFs → write CSVs

In [14]:
def build_dataframe(projects):
    """Convert project list to a DataFrame with the standard column order."""
    if not projects:
        return pd.DataFrame(columns=REQUIRED_COLS)
    df = pd.DataFrame(projects)
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = ''
    extra = [c for c in df.columns if c not in REQUIRED_COLS]
    return df[REQUIRED_COLS + extra]


summary = []

for pdf_path in sorted(PDF_DIR.glob('*.pdf')):
    stem = pdf_path.stem
    print(f'\n══ {pdf_path.name} ══')

    cip_year, projects = parse_pdf(pdf_path)

    df       = build_dataframe(projects)
    out_path = CSV_DIR / f'{stem}.csv'
    df.to_csv(out_path, index=False)

    if not df.empty:
        for dept, n in sorted(df.groupby('department').size().items()):
            print(f'  {dept}: {n}')

    print(f'  Total rows: {len(df)}  → {out_path.name}')
    summary.append({'file': pdf_path.name, 'cip_year': cip_year, 'rows': len(df)})

print('\n\n══ ALL DONE ══')


══ 2007.pdf ══
  AVIATION FACILITIES: 39
  CITY FACILITIES: 267
  CONVENTION AND EVENT SERVICES FACILITIES: 13
  CULTURAL FACILITIES: 50
  ECONOMIC DEVELOPMENT PROGRAMS AND INITIATIVES: 79
  EQUIPMENT ACQUISITION: 37
  FLOOD PROTECTION AND STORM DRAINAGE: 154
  PARK AND RECREATION: 549
  STREET AND THOROUGHFARE: 1755
  TRINITY RIVER CORRIDOR: 20
  WATER UTILITIES: 63
  Total rows: 3026  → 2007.csv

══ 2008.pdf ══
  AVIATION FACILITIES: 41
  CITY FACILITIES: 257
  CONVENTION AND EVENT SERVICES FACILITIES: 16
  CULTURAL FACILITIES: 55
  ECONOMIC DEVELOPMENT PROGRAMS AND INITIATIVES: 91
  EQUIPMENT ACQUISITION: 40
  FLOOD PROTECTION AND STORM DRAINAGE: 150
  PARK AND RECREATION: 528
  STREET AND THOROUGHFARE: 1510
  TRINITY RIVER CORRIDOR: 15
  WATER UTILITIES: 60
  Total rows: 2763  → 2008.csv

══ 2010.pdf ══
  AVIATION FACILITIES: 41
  CITY FACILITIES: 236
  CONVENTION AND EVENT SERVICES FACILITIES: 19
  CULTURAL FACILITIES: 54
  ECONOMIC DEVELOPMENT PROGRAMS AND INITIATIVES: 86
  EQUI

## Cell 9 – Null-rate diagnostics

In [15]:
NULL_MARKERS = {'', 'nan', 'na', 'n/a', 'tbd', 'none'}

DIAG_COLS = [
    'department', 'project_name', 'project_id',
    'project_type', 'funding_source', 'council_district',
    'previous_appropriations', 'project_total',
]

print(f'{"File":<20} {"Rows":>5}  ' +
      '  '.join(f'{c[:14]:>14}' for c in DIAG_COLS))
print('-' * (28 + 16 * len(DIAG_COLS)))

for info in summary:
    csv_file = CSV_DIR / info['file'].replace('.pdf', '.csv')
    df = pd.read_csv(csv_file, dtype=str).fillna('')
    n  = len(df)

    def pct(col):
        if col not in df or n == 0:
            return 'N/A'
        filled = (~df[col].str.strip().str.lower().isin(NULL_MARKERS)).sum()
        return f'{filled}/{n}'

    vals = '  '.join(f'{pct(c):>14}' for c in DIAG_COLS)
    print(f'{info["file"].replace(".pdf",".csv"):<20} {n:>5}  {vals}')

File                  Rows      department    project_name      project_id    project_type  funding_source  council_distri  previous_appro   project_total
------------------------------------------------------------------------------------------------------------------------------------------------------------
2007.csv              3026       3026/3026       3026/3026          0/3026       3026/3026       3026/3026       3026/3026       3026/3026       3026/3026
2008.csv              2763       2763/2763       2763/2763          0/2763       2763/2763       2763/2763       2758/2763       2763/2763       2763/2763
2010.csv              2222       2222/2222       2222/2222          0/2222       2222/2222       2222/2222       2221/2222       2222/2222       2222/2222
2011.csv              1794       1794/1794       1794/1794          0/1794       1794/1794       1794/1794       1790/1794       1794/1794       1794/1794
2012.csv              2034       2034/2034       2034/2034          

## Cell 10 – Spot-checks: verify known projects

In [16]:
CHECK_CASES = [
    # (csv_stem, name_fragment,            field,                    expected)
    # ── 2019 (90° rotation, Format D) ────────────────────────────────────────
    ('2019', 'City Hall',                  'project_id',             'VH05'),
    ('2019', 'City Hall',                  'project_name',           'City Hall'),
    ('2019', 'City Hall',                  'department',             'CITY FACILITIES'),
    ('2019', 'City Hall',                  'project_type',           'City and Municipal Court Facilities'),
    # ── 2025 (0° rotation, Format E) ─────────────────────────────────────────
    ('2025', '2010 CO',                    'project_total',          77462.0),
    ('2025', '2010 CO',                    'project_id',             'IC70'),
    ('2025', 'ADA Improvements',           'project_id',             'W793'),
    ('2025', 'ADA Improvements',           'funding_source',         'Other GO CIP - Non-Debt'),
    # ── 2007 (90° rotation, Format A) ────────────────────────────────────────
    ('2007', 'Cadillac Heights Land Acquisition', 'funding_source',  '06 Bond Program'),
]

print(f'{"CSV":<8} {"Name fragment":<40} {"Field":<25} '
      f'{"Expected":>30} {"Got":>20}  Status')
print('-' * 135)

all_ok = True
for stem, frag, field, expected in CHECK_CASES:
    csv_path = CSV_DIR / f'{stem}.csv'
    if not csv_path.exists():
        print(f'{stem:<8} {frag:<40} – file missing'); all_ok = False; continue

    df  = pd.read_csv(csv_path, dtype=str).fillna('')
    hit = df[df['project_name'].str.lower().str.contains(
        re.escape(frag.lower()), na=False)]

    if hit.empty:
        print(f'{stem:<8} {frag:<40} – NOT FOUND'); all_ok = False; continue

    raw = hit.iloc[0].get(field, '')
    try:
        got = (float(raw) if isinstance(expected, float)
               else int(raw)  if isinstance(expected, int)
               else str(raw))
    except (ValueError, TypeError):
        got = raw

    status = '✓ PASS' if got == expected else '✗ FAIL'
    if '✗' in status:
        all_ok = False
    print(f'{stem:<8} {frag:<40} {field:<25} '
          f'{str(expected):>30} {str(got):>20}  {status}')

print()
print('All spot-checks passed ✓' if all_ok else 'Some checks failed — review output.')

CSV      Name fragment                            Field                                           Expected                  Got  Status
---------------------------------------------------------------------------------------------------------------------------------------
2019     City Hall                                project_id                                          VH05                 T747  ✗ FAIL
2019     City Hall                                project_name                                   City Hall City Hall - Electrical system renovations  ✗ FAIL
2019     City Hall                                department                               CITY FACILITIES      CITY FACILITIES  ✓ PASS
2019     City Hall                                project_type              City and Municipal Court Facilities City and Municipal Court Facilities  ✓ PASS
2025     2010 CO                                  project_total                                    77462.0              77462.0  ✓ PASS
2025   